# 10 Group errors and residuals on the scored year (C-03 test, C-04)

Written by Yiling (2026-11-15). Uses the BD-07 predictions: **test** (2024) once the one-time test has run, otherwise the **dry run** (2023). Thresholds come from the frozen spec (`reports/bd07_frozen_spec.json`) and are not re-chosen. Errors are relative to the historical decision label. Groups with n < `MIN_GROUP_N` are not reported. Judge codes are anonymous; residuals are predictive, not causal.

In [1]:
import sys; sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import config
assert config.ROOT == Path.cwd().resolve().parent, "Run from the notebooks directory."
(config.REPORTS / "figures").mkdir(exist_ok=True)
import json
from src import explain
from src.fairness import group_metrics, residuals
LABEL = explain.available_label()
print("Using:", LABEL)
spec = json.loads((config.REPORTS / "bd07_frozen_spec.json").read_text())
pred = pd.read_parquet(config.DATA_PROCESSED / f"predictions_{LABEL}.parquet")
pred["split"] = "scored"
rows = explain.scored_rows(LABEL)
JUDGE = config.JUDGE_COL_FINAL if LABEL == "test" else "judge_id"
meta = rows[[config.ID_COL, "county_name", JUDGE, *config.FAIRNESS_GROUPS]].rename(columns={JUDGE: "judge_code"})
cuts = pd.read_csv(config.REPORTS / f"bd07_{LABEL}_confusion.csv").set_index(["model", "feature_set"])["threshold"]
print(cuts)

Using: dryrun


model  feature_set     
xgb    CASE                0.352014
       CASE_COURT          0.341572
       CASE_COURT_JUDGE    0.351024
logit  CASE                0.341424
       CASE_COURT          0.340640
       CASE_COURT_JUDGE    0.343132
Name: threshold, dtype: float64


## C-03 Group errors and calibration

In [2]:
tables = []
for (model, fs), t in cuts.items():
    part = pred.loc[pred.model.eq(model) & pred.feature_set.eq(fs)]
    for g in config.FAIRNESS_GROUPS:
        tables.append(group_metrics(part, meta, g, t, split="scored"))
group_table = pd.concat(tables, ignore_index=True)
group_table.to_csv(config.REPORTS / f"c03_{LABEL}_group_metrics.csv", index=False)
show = ["model", "feature_set", "group", "n", "n_pos", "base_rate", "mean_pred", "calibration_gap", "pred_pos_rate", "fpr", "fnr", "roc_auc"]
group_table.loc[group_table.group_col.eq("race") & group_table.model.eq("xgb"), show].round(4)

,model,feature_set,group,n,n_pos,base_rate,mean_pred,calibration_gap,pred_pos_rate,fpr,fnr,roc_auc
0,xgb,CASE,American Indian/Alaskan Native,407,30,0.0737,0.0521,-0.0216,0.0344,0.0186,NaN,NaN
1,xgb,CASE,Asian/Pacific Islander,7219,655,0.0907,0.0711,-0.0196,0.0690,0.0274,0.5145,0.9418
2,xgb,CASE,Black,68765,9840,0.1431,0.1376,-0.0054,0.1529,0.0654,0.3233,0.9301
3,xgb,CASE,Unknown,1768,253,0.1431,0.1124,-0.0307,0.1227,0.0561,0.4783,0.9156
4,xgb,CASE,White,42289,4660,0.1102,0.0966,-0.0136,0.0993,0.0418,0.4367,0.9286
17,xgb,CASE_COURT,American Indian/Alaskan Native,407,30,0.0737,0.0528,-0.0209,0.0344,0.0186,NaN,NaN
18,xgb,CASE_COURT,Asian/Pacific Islander,7219,655,0.0907,0.0697,-0.0210,0.0687,0.0262,0.5053,0.9421
19,xgb,CASE_COURT,Black,68765,9840,0.1431,0.1366,-0.0065,0.1530,0.0644,0.3168,0.9311
20,xgb,CASE_COURT,Unknown,1768,253,0.1431,0.1121,-0.0310,0.1250,0.0554,0.4585,0.9155
21,xgb,CASE_COURT,White,42289,4660,0.1102,0.0964,-0.0138,0.0990,0.0410,0.4324,0.9295


In [3]:
x = group_table.loc[group_table.model.eq("xgb")].set_index(["group_col", "group", "feature_set"])
cols = ["calibration_gap", "pred_pos_rate", "fpr", "fnr"]
change = x.xs("CASE_COURT_JUDGE", level="feature_set")[cols] - x.xs("CASE", level="feature_set")[cols]
change.to_csv(config.REPORTS / f"c03_{LABEL}_judge_change.csv")
change.round(4)

calibration_gap  pred_pos_rate  \
group_col group                                                            
race      American Indian/Alaskan Native          -0.0005         0.0000   
          Asian/Pacific Islander                   0.0015         0.0030   
          Black                                    0.0002        -0.0011   
          Unknown                                  0.0033         0.0090   
          White                                    0.0009         0.0008   
ethnicity Hispanic                                 0.0003        -0.0008   
          Non Hispanic                             0.0006         0.0002   
          Unknown                                  0.0036         0.0057   
gender    Female                                  -0.0005        -0.0005   
          Male                                     0.0008         0.0001   
          Unknown                                  0.0003         0.0011   
age_group 18-24                                   -0.0004        -0.0039   
          25-34                                    0.0004         0.0011   
          35-44                                    0.0016         0.0006   
          45-54                                    0.0002        -0.0021   
          55+                                      0.0007         0.0031   
          <18                                     -0.0004         0.0060   

                                             fpr     fnr  
group_col group                                           
race      American Indian/Alaskan Native -0.0027     NaN  
          Asian/Pacific Islander         -0.0012 -0.0458  
          Black                          -0.0023 -0.0059  
          Unknown                         0.0026 -0.0474  
          White                          -0.0014 -0.0182  
ethnicity Hispanic                       -0.0025 -0.0119  
          Non Hispanic                   -0.0015 -0.0115  
          Unknown                         0.0006 -0.0340  
gender    Female                         -0.0016 -0.0201  
          Male                           -0.0018 -0.0111  
          Unknown                        -0.0031 -0.0368  
age_group 18-24                          -0.0046 -0.0003  
          25-34                          -0.0006 -0.0134  
          35-44                          -0.0008 -0.0099  
          45-54                          -0.0030 -0.0045  
          55+                            -0.0004 -0.0331  
          <18                            -0.0190 -0.0433

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
race = group_table.loc[group_table.group_col.eq("race") & group_table.model.eq("xgb")
                       & group_table.feature_set.isin(["CASE", "CASE_COURT_JUDGE"])]
order = race.loc[race.feature_set.eq("CASE")].sort_values("n", ascending=False)["group"].tolist()
for ax, metric, title in zip(axes, ["fpr", "fnr"], ["False positive rate", "False negative rate"]):
    for k, (fs, marker) in enumerate([("CASE", "o"), ("CASE_COURT_JUDGE", "s")]):
        r = race.loc[race.feature_set.eq(fs)].set_index("group").reindex(order)
        ax.scatter(r[metric], np.arange(len(order)) + (k - 0.5) * 0.25, marker=marker, label=fs)
    ax.set_title(title); ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.grid(axis="x", alpha=0.3)
axes[0].legend(frameon=False, loc="lower right")
fig.suptitle(f"XGBoost, {LABEL}, frozen threshold (relative to historical decisions)")
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / f"c03_{LABEL}_race_errors.png", dpi=200); plt.close(fig)

## C-04 Residuals by county and by anonymous judge code (XGBoost CASE)

Observed rate minus mean predicted probability from the case-and-record model. Positive = more bail/remand than predicted for those cases.

In [5]:
part = pred.loc[pred.model.eq("xgb") & pred.feature_set.eq("CASE")]
county = residuals(part, meta, "county_name", split="scored")
judge = residuals(part, meta, "judge_code", split="scored")
coded = judge.loc[judge["group"].str.startswith("J")].copy()
county.to_csv(config.REPORTS / f"c04_{LABEL}_county_residuals.csv", index=False)
coded.to_csv(config.REPORTS / f"c04_{LABEL}_judge_residuals.csv", index=False)
summary = pd.DataFrame({
    "coded_judges_reported": [len(coded)],
    "share_ci_above_0": [(coded["ci_low"] > 0).mean()],
    "share_ci_below_0": [(coded["ci_high"] < 0).mean()],
    "residual_p10": [coded["residual"].quantile(0.1)], "residual_p90": [coded["residual"].quantile(0.9)],
})
summary.to_csv(config.REPORTS / f"c04_{LABEL}_judge_residual_summary.csv", index=False)
display(county.round(4)); summary.round(4)

,model,feature_set,by,group,n,observed_rate,mean_pred,residual,ci_low,ci_high
0,xgb,CASE,county_name,Bronx,20938,0.1162,0.1372,-0.0210,-0.0243,-0.0176
1,xgb,CASE,county_name,Kings,38662,0.1182,0.1198,-0.0017,-0.0042,0.0008
2,xgb,CASE,county_name,New York,25673,0.1473,0.1242,0.0231,0.0198,0.0265
3,xgb,CASE,county_name,Queens,28908,0.1376,0.1021,0.0355,0.0325,0.0385
4,xgb,CASE,county_name,Richmond,6276,0.1080,0.1020,0.0060,-0.0002,0.0115


,coded_judges_reported,share_ci_above_0,share_ci_below_0,residual_p10,residual_p90
0,115,0.3391,0.2174,-0.0458,0.0757


In [6]:
r = coded.sort_values("residual").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(np.arange(len(r)), r["residual"], yerr=[r["residual"] - r["ci_low"], r["ci_high"] - r["residual"]],
            fmt="o", ms=2.5, lw=0.6, color="#4C72B0", ecolor="#9DB4D9")
ax.axhline(0, color="k", lw=0.7)
ax.set_xlabel("Anonymous judge codes (sorted)"); ax.set_ylabel("Observed − predicted")
ax.set_title(f"Judge residuals, XGBoost CASE, {LABEL} (95% bootstrap intervals)")
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / f"c04_{LABEL}_judge_residuals.png", dpi=200); plt.close(fig)
print("Done. No judge names are used anywhere.")

Done. No judge names are used anywhere.
